# LTO句子的嵌入

为每个LTO题项选择10个意思相近的古文句子 `data/LTO_sentences.csv`，检查这十个句子是否语义相近。

首先使用 qwen3-embedding 对4题项原文和40古文句子嵌入，检查每个题项和对应的十个古文句子是否在同一个聚类。然后使用 bert-ancient-chinese 对40古文句子进行嵌入，看古文句子在专门的古文模型是否仍形成聚类。然后输出 t-sne 降维后的2维坐标，用于绘图。

## 导入与配置

In [ ]:
import numpy as np
from IPython.display import display
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# 需要嵌入的LTO句子
SENT_CSV = Path("../data/LTO_sentences.csv")

# 导出句子向量的路径
OUT_NPY = Path("../data/LTOitem_embeddings/lto_qwen3_embeddings.npy")
OUT_NPY_ANCIENT = Path("../data/LTOitem_embeddings/lto_ancient_embeddings.npy")

# 导出t-sne降维坐标的路径
OUT_CSV = Path("../data/LTOitem_embeddings/lto_qwen3_tsne.csv")
OUT_CSV_ANCIENT = Path("../data/LTOitem_embeddings/lto_ancient_tsne.csv")

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MODEL_NAME_ANCIENT = "Jihuai/bert-ancient-chinese"

BATCH_SIZE = 8

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"使用 {DEVICE}")

## 古文的当代文题项共同嵌入

In [ ]:
model = SentenceTransformer(
    MODEL_NAME, 
    device=DEVICE,                               
    model_kwargs={"attn_implementation": "eager", "torch_dtype": "float32"}
)
model.eval()

df = pd.read_csv(SENT_CSV)

texts = df["句子"].tolist()
all_embeddings = []
with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        embs = model.encode(batch, convert_to_numpy=True)
        all_embeddings.append(embs)
        print(f"  Batch {i // BATCH_SIZE + 1}/{(len(texts) - 1) // BATCH_SIZE + 1}")

# 保存向量文件
embeddings = np.vstack(all_embeddings)
np.save(OUT_NPY, embeddings)
print(f"Saved embeddings: {OUT_NPY}")

## 检查当代文题项和古文的聚类

In [ ]:
# KMeans 聚类（全部样本）
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
embeddings = np.load(OUT_NPY)
df["cluster"] = kmeans.fit_predict(embeddings)

# 构建一个合并所有簇的表格
cluster_dfs = []
for c in range(n_clusters):
    cluster_df = df[df["cluster"] == c][["句子", "模块"]].copy()
    cluster_df.insert(0, "簇", c)  # 添加簇标签列
    cluster_dfs.append(cluster_df)
all_clusters_df = pd.concat(cluster_dfs, ignore_index=True)
display(all_clusters_df)

## 保存古文和当代文的tsne坐标

In [ ]:
tsne_2d = TSNE(n_components=2, perplexity=10, random_state=42, init='pca')
embeddings = np.load(OUT_NPY)
emb_2d = tsne_2d.fit_transform(embeddings)

# 保存 2D 坐标
df_2d = pd.DataFrame({
    "x": emb_2d[:, 0],
    "y": emb_2d[:, 1],
    "label": df["模块"],
    "is_guwen": df["is_guwen"],
    "text": df["句子"],
})

items = df[df["is_guwen"] == False]
mapping = dict(zip(items['模块'], items['句子']))
df_2d["items"] = df_2d["label"].map(mapping)

df_2d.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved 2D coordinates to {OUT_CSV}")

## 只有古文的嵌入

In [ ]:
model_ancient = SentenceTransformer(
    MODEL_NAME_ANCIENT, 
    device=DEVICE,                               
    model_kwargs={"attn_implementation": "eager", "torch_dtype": "float32"}
)
model_ancient.eval()

df = pd.read_csv(SENT_CSV)
df_ancient = df[df["is_guwen"] == True]

texts_ancient = df_ancient["句子"].tolist()
all_embeddings_ancient = []
with torch.no_grad():
    for i in range(0, len(texts_ancient), BATCH_SIZE):
        batch = texts_ancient[i:i + BATCH_SIZE]
        ancient_embs = model_ancient.encode(batch, convert_to_numpy=True)
        all_embeddings_ancient.append(ancient_embs)
        print(f"  Batch {i // BATCH_SIZE + 1}/{(len(texts_ancient) - 1) // BATCH_SIZE + 1}")

embeddings_ancient = np.vstack(all_embeddings_ancient)
np.save(OUT_NPY_ANCIENT, embeddings_ancient)
print(f"Saved embeddings: {OUT_NPY_ANCIENT}")

## 检查古文聚类

In [ ]:
# KMeans 聚类（只有古文）
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
embeddings_ancient = np.load(OUT_NPY_ANCIENT)
df_ancient["cluster"] = kmeans.fit_predict(embeddings_ancient)

# 构建一个合并所有簇的表格
cluster_dfs = []
for c in range(n_clusters):
    cluster_df = df_ancient[df_ancient["cluster"] == c][["句子", "模块"]].copy()
    cluster_df.insert(0, "簇", c)  # 添加簇标签列
    cluster_dfs.append(cluster_df)
all_clusters_df = pd.concat(cluster_dfs, ignore_index=True)
display(all_clusters_df)

# 保存古文的tsne坐标

In [ ]:
tsne_2d_ancient = TSNE(n_components=2, perplexity=10, random_state=42, init='pca')
embeddings_ancient = np.load(OUT_NPY_ANCIENT)
emb_2d_ancient = tsne_2d_ancient.fit_transform(embeddings_ancient)

# 保存 2D 坐标
df_2d_ancient = pd.DataFrame({
    "x": emb_2d_ancient[:, 0],
    "y": emb_2d_ancient[:, 1],
    "label": df_ancient["模块"],
    "is_guwen": df_ancient["is_guwen"],
    "text": df_ancient["句子"],
})

items = df[df["is_guwen"] == False]
mapping = dict(zip(items['模块'], items['句子']))
df_2d_ancient["items"] = df_2d_ancient["label"].map(mapping)

df_2d_ancient.to_csv(OUT_CSV_ANCIENT, index=False, encoding="utf-8")
print(f"Saved 2D coordinates to {OUT_CSV_ANCIENT}")